In [1]:
# Import requested libraries.
import polars as pl
from datetime import datetime
from datetime import date
from datetime import time

In [2]:
# Read excel file.
df = pl.read_excel(r"../data/raw/online_retail_.xlsx")

In [3]:
# Check for general info about the dataset.
df.glimpse()

Rows: 525461
Columns: 8
$ Invoice              <i64> 489434, 489434, 489434, 489434, 489434, 489434, 489434, 489434, 489435, 489435
$ StockCode            <str> '85048', '79323P', '79323W', '22041', '21232', '22064', '21871', '21523', '22350', '22349'
$ Description          <str> '15CM CHRISTMAS GLASS BALL 20 LIGHTS', 'PINK CHERRY LIGHTS', ' WHITE CHERRY LIGHTS', 'RECORD FRAME 7" SINGLE SIZE ', 'STRAWBERRY CERAMIC TRINKET BOX', 'PINK DOUGHNUT TRINKET POT ', 'SAVE THE PLANET MUG', 'FANCY FONT HOME SWEET HOME DOORMAT', 'CAT BOWL ', 'DOG BOWL , CHASING BALL DESIGN'
$ Quantity             <i64> 12, 12, 12, 48, 24, 24, 24, 10, 12, 12
$ InvoiceDate <datetime[ms]> 2009-12-01 07:45:00, 2009-12-01 07:45:00, 2009-12-01 07:45:00, 2009-12-01 07:45:00, 2009-12-01 07:45:00, 2009-12-01 07:45:00, 2009-12-01 07:45:00, 2009-12-01 07:45:00, 2009-12-01 07:46:00, 2009-12-01 07:46:00
$ Price                <f64> 6.95, 6.75, 6.75, 2.1, 1.25, 1.65, 1.25, 5.95, 2.55, 3.75
$ Customer ID          <i64> 13085, 13

In [4]:
# Count of duplicate records.
df.is_duplicated().sum()

13283

In [5]:
# Check for null values.
df.null_count().sum()

Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
u32,u32,u32,u32,u32,u32,u32,u32
10209,0,2928,0,0,0,107927,0


In [6]:
# Rename columns.
df = df.rename({
     "Invoice" : "invoice",
    "StockCode" : "stock_code",
    "Description" : "description",
    "InvoiceDate" : "invoice_date_time",
    "Quantity" : "quantity",
    "Price" : "price",
    "Customer ID" : "customer_id",
    "Country" : "country",
}
)

In [7]:
# Separate invoice date and time column into date and time columns.
df = df.with_columns(pl.col("invoice_date_time").dt.date().alias("invoice_date"))
df = df.with_columns(pl.col("invoice_date_time").dt.time().alias("invoice_time"))

In [8]:
# Identify exact duplicate rows and count their occurrences.
duplicate_rows = df.group_by(df.columns).len(name = "count of repeated records").filter(pl.col("count of repeated records") > 1).sort(by = "count of repeated records", descending = True)
duplicate_rows

invoice,stock_code,description,quantity,invoice_date_time,price,customer_id,country,invoice_date,invoice_time,count of repeated records
i64,str,str,i64,datetime[ms],f64,i64,str,date,time,u32
534219,"""35953""","""FOLKART STAR CHRISTMAS DECORAT…",1,2010-11-21 16:13:00,1.25,13230,"""United Kingdom""",2010-11-21,16:13:00,6
496431,"""84826""","""ASSTD DESIGN 3D PAPER STICKERS""",1,2010-02-01 12:30:00,0.85,16415,"""United Kingdom""",2010-02-01,12:30:00,6
502660,"""17021""","""NAMASTE SWAGAT INCENSE""",6,2010-03-25 17:18:00,0.3,13187,"""United Kingdom""",2010-03-25,17:18:00,6
525065,"""20894""","""HANGING BAUBLE T-LIGHT HOLDER …",1,2010-10-03 14:28:00,2.95,16799,"""United Kingdom""",2010-10-03,14:28:00,6
530953,"""21231""","""SWEETHEART CERAMIC TRINKET BOX""",4,2010-11-04 18:42:00,1.25,18168,"""United Kingdom""",2010-11-04,18:42:00,5
…,…,…,…,…,…,…,…,…,…,…
528470,"""85017A""","""ENVELOPE 50 ROMANTIC IMAGES""",1,2010-10-22 11:40:00,0.85,16392,"""United Kingdom""",2010-10-22,11:40:00,2
527262,"""21888""","""BINGO SET""",1,2010-10-15 12:45:00,3.75,15574,"""United Kingdom""",2010-10-15,12:45:00,2
508864,"""20971""","""PINK BLUE FELT CRAFT TRINKET B…",1,2010-05-18 16:09:00,1.25,17589,"""United Kingdom""",2010-05-18,16:09:00,2


In [9]:
# Verify whether identified duplicate records are exact matches.
df.filter((pl.col("customer_id") == 16799) & (pl.col("invoice_time") == time(14,28,0)) & (pl.col("stock_code") == "20894"))

invoice,stock_code,description,quantity,invoice_date_time,price,customer_id,country,invoice_date,invoice_time
i64,str,str,i64,datetime[ms],f64,i64,str,date,time
525065,"""20894""","""HANGING BAUBLE T-LIGHT HOLDER …",1,2010-10-03 14:28:00,2.95,16799,"""United Kingdom""",2010-10-03,14:28:00
525065,"""20894""","""HANGING BAUBLE T-LIGHT HOLDER …",1,2010-10-03 14:28:00,2.95,16799,"""United Kingdom""",2010-10-03,14:28:00
525065,"""20894""","""HANGING BAUBLE T-LIGHT HOLDER …",1,2010-10-03 14:28:00,2.95,16799,"""United Kingdom""",2010-10-03,14:28:00
525065,"""20894""","""HANGING BAUBLE T-LIGHT HOLDER …",1,2010-10-03 14:28:00,2.95,16799,"""United Kingdom""",2010-10-03,14:28:00
525065,"""20894""","""HANGING BAUBLE T-LIGHT HOLDER …",1,2010-10-03 14:28:00,2.95,16799,"""United Kingdom""",2010-10-03,14:28:00
525065,"""20894""","""HANGING BAUBLE T-LIGHT HOLDER …",1,2010-10-03 14:28:00,2.95,16799,"""United Kingdom""",2010-10-03,14:28:00


In [10]:
# Calculate duplicate rows and percentage of duplicate records.
duplicate_percentage = (df.is_duplicated().sum()/df.shape[0]) * 100 
print(f"Duplicate rows : {df.is_duplicated().sum()}")
print(f"Percentage of duplicate rows : {duplicate_percentage:.2f} %")

Duplicate rows : 13283
Percentage of duplicate rows : 2.53 %


In [11]:
# Remove exact duplicate rows from the DataFrame.
df = df.unique()

In [12]:
# Remove rows with missing customer IDs.
df = df.drop_nulls(subset = "customer_id")

In [13]:
# Check for remaining exact duplicate rows.
print("Duplicate rows:", df.is_duplicated().sum())

# Check missing values in each column.
print(df.null_count())

# Check the final dataset dimensions.
print("Final shape:", df.shape)

Duplicate rows: 0
shape: (1, 10)
┌─────────┬────────────┬─────────────┬──────────┬───┬─────────────┬─────────┬──────────────┬──────────────┐
│ invoice ┆ stock_code ┆ description ┆ quantity ┆ … ┆ customer_id ┆ country ┆ invoice_date ┆ invoice_time │
│ ---     ┆ ---        ┆ ---         ┆ ---      ┆   ┆ ---         ┆ ---     ┆ ---          ┆ ---          │
│ u32     ┆ u32        ┆ u32         ┆ u32      ┆   ┆ u32         ┆ u32     ┆ u32          ┆ u32          │
╞═════════╪════════════╪═════════════╪══════════╪═══╪═════════════╪═════════╪══════════════╪══════════════╡
│ 9816    ┆ 0          ┆ 0           ┆ 0        ┆ … ┆ 0           ┆ 0       ┆ 0            ┆ 0            │
└─────────┴────────────┴─────────────┴──────────┴───┴─────────────┴─────────┴──────────────┴──────────────┘
Final shape: (410763, 10)


In [14]:
# Separate records with missing invoice numbers and negative quantities for further investigation.
df_negative_qunatity = df.filter(pl.col("quantity") < 0)
df_negative_qunatity

invoice,stock_code,description,quantity,invoice_date_time,price,customer_id,country,invoice_date,invoice_time
i64,str,str,i64,datetime[ms],f64,i64,str,date,time
null,"""22295""","""HEART FILIGREE DOVE LARGE""",-12,2010-08-20 09:25:00,1.65,14064,"""United Kingdom""",2010-08-20,09:25:00
null,"""21734""","""SMALL MARAKESH LANTERN ON STIC…",-4,2010-03-22 15:44:00,3.75,15791,"""United Kingdom""",2010-03-22,15:44:00
null,"""22325""","""MOBILE VINTAGE HEARTS """,-2,2010-05-14 11:04:00,4.95,13711,"""United Kingdom""",2010-05-14,11:04:00
null,"""82607A""","""GLASS BONBON JAR PARFUME AUX F…",-2,2010-03-03 15:15:00,1.25,13797,"""United Kingdom""",2010-03-03,15:15:00
null,"""21143""","""ANTIQUE GLASS HEART DECORATION…",-4,2010-11-02 10:03:00,1.95,17044,"""United Kingdom""",2010-11-02,10:03:00
…,…,…,…,…,…,…,…,…,…
null,"""22699""","""ROSES REGENCY TEACUP AND SAUCE…",-6,2010-12-09 13:38:00,2.95,14205,"""United Kingdom""",2010-12-09,13:38:00
null,"""35004C""","""SET OF 3 COLOURED FLYING DUCK…",-1,2010-05-10 14:03:00,4.65,15311,"""United Kingdom""",2010-05-10,14:03:00
null,"""22865""","""HAND WARMER OWL DESIGN""",-10,2010-12-06 13:12:00,2.1,15882,"""United Kingdom""",2010-12-06,13:12:00


In [15]:
# Write records with missing invoice numbers and negative quantities to another csv file for further investigation.
df_negative_qunatity.write_csv("records_missing_invoice_negative_quantity.csv")

In [16]:
# Write cleaned dataset to a csv.
df.write_csv("online_retail_clean.csv")